# DE Results Analysis: Cross-Variant Comparison

This notebook processes the Differential Expression (DE) results from all 8 variants of the analysis.
We aim to:
1. Identify how many genes are significantly differentially expressed (`padj < 0.05`) in each variant.
2. Compare the ratio of significant genes to the background of valid genes.
3. Analyze the intersection/overlap of these significant genes across the different analyses.

In [ ]:
import os
import glob
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

# Optional: set seaborn style for nicer plots
sns.set_theme(style="whitegrid")

In [ ]:
# Define the root directory for DE results
de_dir = '/Users/ppopov1/_circRNA/RNAseq/04_DE_Analysis_results'

# Find all DE result CSV files using a glob pattern
search_pattern = os.path.join(de_dir, '*', '*_DE_results_*.csv')
de_files = glob.glob(search_pattern)

print(f"Found {len(de_files)} DE result files.")

results_summary = []
all_significant_genes = []

for file_path in sorted(de_files):
    # Extract a clean name for the analysis variant from the parent folder name
    folder_name = os.path.basename(os.path.dirname(file_path))
    analysis_name = folder_name.split('-', 1)[-1] if '-' in folder_name else folder_name
    
    # Load the CSV
    df = pd.read_csv(file_path)
    
    # Filter out rows where 'padj' is NA (these are genes with extremely low counts ignored by DESeq2)
    valid_genes_df = df.dropna(subset=['padj'])
    total_valid = len(valid_genes_df)
    
    # Filter for statistically significant genes
    sig_df = valid_genes_df[valid_genes_df['padj'] < 0.05]
    sig_count = len(sig_df)
    
    # Calculate percentage
    ratio = (sig_count / total_valid * 100) if total_valid > 0 else 0
    
    results_summary.append({
        'Analysis Variant': analysis_name,
        'Total Valid Genes': total_valid,
        'Significant Genes': sig_count,
        '% Significant': round(ratio, 2)
    })
    
    # Extract unique gene names for overlap tracking
    # (We use 'gene_name' as this is what matches the circRNA matrix HGNC symbols)
    sig_gene_names = sig_df['gene_name'].dropna().unique()
    all_significant_genes.extend(sig_gene_names)
    
# Display the summary table
summary_df = pd.DataFrame(results_summary)
display(summary_df)

In [ ]:
# Count how many analyses each unique gene appeared in
gene_counts = Counter(all_significant_genes)

# Convert the Counter dictionary to a Pandas DataFrame
overlap_df = pd.DataFrame.from_dict(gene_counts, orient='index', columns=['Appearance_Count'])
overlap_df.index.name = 'Gene'
overlap_df = overlap_df.reset_index()

# Tally the frequencies (how many genes appeared 1 time, 2 times, etc.)
frequency_distribution = overlap_df['Appearance_Count'].value_counts().sort_index()

print("Gene Overlap Frequency Distribution:")
print(frequency_distribution)

In [ ]:
import pandas as pd
import os
import glob

de_dir = '/Users/ppopov1/_circRNA/RNAseq/04_DE_Analysis_results'
search_pattern = os.path.join(de_dir, '*', '*_DE_results_*.csv')
de_files = glob.glob(search_pattern)

all_sig_genes_data = []

# Loop through the files, using 'enumerate' to get our numeric code (0 to 7)
for i, file_path in enumerate(sorted(de_files)):
    folder_name = os.path.basename(os.path.dirname(file_path))
    analysis_name = folder_name.split('-', 1)[-1] if '-' in folder_name else folder_name
    
    # Numeric code for this analysis
    analysis_code = i
    
    df = pd.read_csv(file_path)
    
    # Filter strictly for significant genes
    sig_df = df[df['padj'] < 0.05].copy()
    
    if not sig_df.empty:
        # Inject our custom tracking columns
        sig_df['analysis_name'] = analysis_name
        sig_df['analysis_code'] = analysis_code
        
        # Select and reorder exactly the columns we discussed
        cols_to_keep = [
            'gene_name', 'gene_id', 'gene_type', 
            'analysis_name', 'analysis_code', 
            'padj', 'log2FoldChange', 'baseMean'
        ]
        
        all_sig_genes_data.append(sig_df[cols_to_keep])

# Combine all the data into one massive master table
master_sig_df = pd.concat(all_sig_genes_data, ignore_index=True)

# Save the final CSV to the data directory
output_csv = '/Users/ppopov1/_circRNA/docs/01_on_data/data/significant_DE_genes.csv'
master_sig_df.to_csv(output_csv, index=False)

print(f"Master CSV successfully saved with {len(master_sig_df)} significant gene entries!")
display(master_sig_df.head(10))
